# Sections 1 and 2: Positioning and Direct Replication

**ECON 5200 Midterm Project · Lee (2008), U.S. House Elections**

Paper: David S. Lee (2008), *Randomized Experiments from Non-random Selection in U.S. House Elections*, *Journal of Econometrics*, 142(2), 675–697.

**Important:** The supplied `lmb-data.dta` comes from Lee, Moretti & Butler (2004), not Lee's original 1946–1998 analysis file. It covers 1948–1990. This notebook estimates the assigned-data discontinuity and investigates—not assumes—the reasons for disagreement with the paper's 0.077 headline.

Run cells from the top. The report itself follows the separate Word template.

## Setup

Imports and paths only.

In [ ]:
# Setup — works inside the GitHub template and when opened separately in Colab
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

CWD = Path.cwd()
if (CWD / "download_data.py").exists():
    ROOT = CWD
elif (CWD.parent / "download_data.py").exists():
    ROOT = CWD.parent
else:
    ROOT = CWD  # standalone notebook / Colab session

sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(parents=True, exist_ok=True)

# Optional: use the template's helper if this notebook is in a full repository.
try:
    from download_data import MY_PAPER, PAPERS, fetch
    HAS_REPO_HELPER = True
    print("Repository download_data.py detected.")
except ModuleNotFoundError as exc:
    if exc.name != "download_data":
        raise
    HAS_REPO_HELPER = False
    MY_PAPER = "lee"
    print("Standalone mode: no download_data.py; will load the public Lee/LMB file directly.")

print("Output folder:", ROOT / "output")

## Data

In [ ]:
# Data — repo-aware, with a standalone Colab fallback
# Source supplied by the assignment brief: Lee, Moretti & Butler (2004).
DATA_URL = "https://raw.githubusercontent.com/scunning1975/mixtape/master/lmb-data.dta"

# First try an existing local .dta (for offline use), then the repository's fetch
# utility (when configured for Lee), and finally the public URL.
local_candidates = [ROOT / "data" / "lmb-data.dta", ROOT / "lmb-data.dta", CWD / "lmb-data.dta"]
local_file = next((p for p in local_candidates if p.is_file()), None)

if local_file is not None:
    df = pd.read_stata(local_file)
    print("Data loaded from:", local_file)
elif HAS_REPO_HELPER and MY_PAPER == "lee":
    try:
        paths = fetch(MY_PAPER)
        df = pd.read_stata(paths[0])
        print("Data loaded through download_data.py:", paths[0])
    except Exception as exc:
        print("Repository fetch failed; trying the public data URL:", str(exc))
        df = pd.read_stata(DATA_URL)
        print("Data loaded from:", DATA_URL)
else:
    try:
        df = pd.read_stata(DATA_URL)
        print("Data loaded from:", DATA_URL)
    except Exception as exc:
        raise RuntimeError(
            "Unable to download lmb-data.dta. Upload it to the current folder "
            "or place it at data/lmb-data.dta, then rerun this cell."
        ) from exc

required = ["demvoteshare", "lagdemvoteshare", "year", "state", "district"]
missing = [v for v in required if v not in df.columns]
if missing:
    raise KeyError(f"Missing expected columns: {missing}. Available: {df.columns.tolist()}")

# 'demvoteshare' is the outcome, and the lagged share determines treatment.
lmb = df.dropna(subset=["demvoteshare", "lagdemvoteshare"]).copy()
lmb["margin"] = lmb["lagdemvoteshare"].astype(float) - 0.5
CUTOFF = 0.0

print("Raw observations:", len(df))
print("Nonmissing outcome and lag:", len(lmb))
print("Year range in file:", int(lmb["year"].min()), "to", int(lmb["year"].max()))
print("Outcome: demvoteshare; running variable: lagdemvoteshare - 0.5")
print("Cutoff:", CUTOFF)

## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. **Caughey & Sekhon (2011)**, “Elections and the Regression Discontinuity Design: Lessons from Close U.S. House Races, 1942–2008” (*Political Analysis*). They document differences in pre-election characteristics of narrow winners and losers, challenging the assumption that every close House election is locally comparable.
2. **Eggers et al. (2015)**, “On the Validity of the Regression Discontinuity Design for Estimating Electoral Effects: New Evidence from Over 40,000 Close Races” (*American Journal of Political Science*). They examine many other close elections and find that the problematic U.S. House patterns do not generalize widely.
3. **de la Cuesta & Imai (2016)**, “Misunderstandings about the Regression Discontinuity Design in the Study of Close Elections” (*Annual Review of Political Science*). They clarify how a continuity-based RD differs from stronger local-randomization claims and discuss interpretation of balance/sorting tests.

**The identification strategy and its assumption, in my own words:**

Lee (2008) uses **sharp regression discontinuity** to compare districts where Democrats barely won versus barely lost the previous election. The running variable is the Democratic margin in the previous race; winning at the zero cutoff changes which party holds the seat. The main outcome is the Democratic party's vote share in the following election. If the potential outcome trends would have been smooth at zero without winning, the jump in next-election vote share is the causal effect of party incumbency for close elections. An important threat is exact sorting or manipulation of close-election outcomes. Predetermined covariates should also look similar immediately across the cutoff if the strong local-randomization account applies.

**Position of my replication:** The assigned LMB file covers 1948–1990 and uses a two-party vote-share running variable. Lee (2008) used a longer electoral series and a margin over the strongest opponent. My Section 2 compares the estimated jump with the published **0.077** without claiming that the year difference alone explains any gap.

*The full prose for Section 1 belongs in the written report; these are the notebook research notes.*

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [ ]:
# Section 2: direct replication — global quartic, separate curves on either side
# 'H=0.5' covers the feasible margin range for a two-party vote share.
# This reproduces the baseline assigned-data exercise, not Lee's original raw sample.
H = 0.5
KERNEL = "rectangular"  # uniform kernel: equal weights inside the window
ORDER = 4

sample = lmb.dropna(subset=["demvoteshare", "margin", "year", "state", "district"]).copy()
sample = sample.loc[sample["margin"].abs() <= H].copy()
sample["win"] = (sample["margin"] >= CUTOFF).astype(int)

x = sample["margin"].astype(float)
X = pd.DataFrame({"const": 1.0, "win": sample["win"].astype(float)}, index=sample.index)
for p in range(1, ORDER + 1):
    X[f"margin_{p}"] = x ** p
    X[f"win_margin_{p}"] = sample["win"] * (x ** p)

sample["decade"] = (sample["year"].astype(int) // 10) * 10
cluster = pd.factorize(
    sample[["state", "district", "decade"]].astype(str).agg("_".join, axis=1)
)[0]

# WLS with uniform weights is OLS; cluster covariance at state/district/decade level.
rd_fit = sm.WLS(sample["demvoteshare"].astype(float), X,
                weights=np.ones(len(sample))).fit(
    cov_type="cluster", cov_kwds={"groups": cluster}
)
JUMP = float(rd_fit.params["win"])
JUMP_SE = float(rd_fit.bse["win"])
N = int(rd_fit.nobs)
CI_LOW = JUMP - 1.96 * JUMP_SE
CI_HIGH = JUMP + 1.96 * JUMP_SE
PAPER_JUMP = 0.077
PAPER_SE = 0.011
PAPER_N = 6558

print(f"Our RDD jump: {JUMP:.4f} = {100 * JUMP:.2f} percentage points")
print(f"Clustered SE: {JUMP_SE:.4f} ({100 * JUMP_SE:.2f} percentage points)")
print(f"95% CI: [{CI_LOW:.4f}, {CI_HIGH:.4f}]")
print(f"Sample observations: {N:,}; clusters: {len(np.unique(cluster)):,}")
print(f"Lee (2008), Table 2 column (1): {PAPER_JUMP:.3f} ({100*PAPER_JUMP:.2f} pp)")
print(f"Gap: {JUMP - PAPER_JUMP:+.4f} = {100*(JUMP - PAPER_JUMP):+.2f} pp")
print("\nCounts by observed outcome year:")
print(sample.groupby("year").size().to_string())

# Additional diagnostic, NOT a replacement for the assigned-data baseline:
# Lee (2008) reports excluding running-variable years ending in 0 or 2.
# The file's year labels the outcome election, so running-variable year is year - 2.
running_year = sample["year"].astype(int) - 2
keep_year_rule = ~running_year.mod(10).isin([0, 2])
print(f"\nDiagnostic: remaining after removing running years ending 0/2: {int(keep_year_rule.sum()):,}")
print("This count alone does not reconstruct Lee's original sample.")

### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [ ]:
# Paper-versus-replication table: all 'my' values come from Cell 9.
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)

headline.loc["Table 2, col (1): incumbency jump (share units)"] = [
    PAPER_JUMP, JUMP, np.nan,
    "Different years, sample restrictions, running-variable definition"
]
headline.loc["Table 2, col (1): clustered SE (share units)"] = [
    PAPER_SE, JUMP_SE, np.nan,
    "Different samples and clustering composition"
]
headline.loc["Table 2, col (1): observations"] = [
    PAPER_N, N, np.nan,
    "Different sample coverage and original paper's sample construction"
]

headline["gap_pct"] = (100 *
    (headline["my_estimate"] - headline["paper_estimate"])
    / headline["paper_estimate"].abs()
)
headline.to_csv(ROOT / "output" / "headline_table.csv")
display(headline)
print("Saved:", ROOT / "output" / "headline_table.csv")
print(f"Headline difference (percentage points): {100 * (JUMP - PAPER_JUMP):+.2f}")

# A compact table with explicit units for the written report.
report_table = pd.DataFrame({
    "Measure": ["Incumbency effect (percentage points)",
                "Clustered SE (percentage points)", "Observations"],
    "Lee (2008) Table 2 (1)": [100 * PAPER_JUMP, 100 * PAPER_SE, PAPER_N],
    "Assigned LMB replication": [100 * JUMP, 100 * JUMP_SE, N],
})
report_table["Gap"] = report_table["Assigned LMB replication"] - report_table["Lee (2008) Table 2 (1)"]
report_table.to_csv(ROOT / "output" / "headline_report_table.csv", index=False)
print("\nReport-ready summary:")
display(report_table)

### Discrepancy log

**Important:** Numeric differences below refer to a *previous* run of the same baseline specification (**0.0861**, SE **0.0098**, **N = 13,566**). The current run prints the authoritative values in Cells 9 and 11. Update this narrative if your current output changes.

| # | Estimate (row above) | What differs | Size of gap from previous run | Likely source | How I checked it | Status / ambiguity |
|---|---|---|---|---|---|---|
| 1 | Table 2 (1): incumbency jump | LMB estimate exceeds Lee (2008) | +0.0091 share units (+0.91 percentage points) | Distinct year coverage; possibly sample and variable construction | Compared reported coverage and actual specification; Cell 9 prints year counts | **Unresolved:** cannot isolate missing-year contribution without Lee's full underlying data |
| 2 | Table 2 (1): clustered SE | Standard errors differ | −0.0012 share units | Different number of observations and clusters | Cell 9 prints cluster count; compare to published SE | **Unresolved:** exact cluster compositions differ |
| 3 | Table 2 (1): observations | LMB model has more rows | +7,008 | Lee's election-pair restrictions / sample construction | Cell 9 prints counts and a diagnostic based on the exclusion of running years ending 0 or 2 | **Unresolved:** year coverage alone does not explain the row difference |
| 4 | Running variable definition | Constructed LMB margin differs conceptually from the paper's strongest-opponent margin | Not separately identified | `lagdemvoteshare - 0.5` vs vote-share margin over strongest opponent | Compared the starter variables with Lee (2008), §3.3 | **Unresolved:** cannot assume equivalence when third-party votes matter |

**Interpretation.** The assigned data reproduce a positive party-incumbency advantage close to nine percentage points, compared with Lee's 7.7 percentage points. The difference does not reverse the sign of the result. But a similar sign is not proof that the full original specification was reproduced: the observation count, election years, and running variable are not identical. The assignment clarification suggests missing years as a first place to investigate, **not** as an established causal explanation of the gap. The current code produces the exact gap and diagnostics; any final report should copy the current output, not these earlier rounded values.

**Inference note.** The replication uses heteroskedasticity-robust covariance clustered on state–district–decade combinations. This is an implementation of the paper's stated clustering level, but identical underlying clusters cannot be guaranteed with the assigned data.

**Ambiguities and specification choices to investigate in Section 3:**

- **Bandwidth:** compare an MSE-selected bandwidth with narrower and wider windows; check whether the positive jump persists.
- **Polynomial order:** contrast the global fourth-order polynomial with local linear and local quadratic fits. High-order global polynomials can be sensitive to observations far from the cutoff.
- **Density / sorting:** test whether the running-variable distribution is smooth at zero, bearing in mind the published debate about very close House races.
- **Donut RD and placebo cutoffs:** omit observations closest to zero, and test artificial thresholds where the treatment rule does not change.
- **Year coverage / heterogeneous effects:** examine estimates by decade among available years. This can reveal changes over time, but cannot recover years missing from LMB.
- **Baseline covariates:** where available, test whether characteristics determined before the election have discontinuities at the cutoff.

**For every Section 3 exercise, record:** what changed, numerical estimate and interval, and what it implies for Lee's causal claim.